# TP3 — Analisis de sentimiento en tweets

**Diplomatura en Inteligencia Artificial — Marcelo Vieira**

Esta es la **notebook principal** del TP3: resume el problema, los datos, las
decisiones, los resultados y las conclusiones. El desarrollo completo esta en las
6 notebooks de [`notebooks/`](./notebooks/), enlazadas al final.

---

## 1. El problema

Dado el texto de un tweet, predecir si expresa un sentimiento **negativo o positivo**.

- Tarea de NLP: clasificacion supervisada de texto.
- Dataset: **Sentiment140** — 1.600.000 tweets en ingles etiquetados automaticamente
  (por emoticones), mas 498 tweets etiquetados **a mano** como test.
- Requisito mandatorio de la consigna: los resultados finales se calculan sobre el
  **dataset completo**, sin muestras.

## 2. Los datos y sus tres rarezas

![Distribucion de clases](imgs/02_distribucion_clases.png)

1. **Training balanceado perfecto**: 800.000 negativos y 800.000 positivos.
2. **El training no tiene clase neutral** (el test manual si) -> el modelo principal
   es **binario**; neutral se trata como extension exploratoria.
3. **El archivo viene ordenado por clase** (primero todos los negativos) -> cualquier
   split se hace con mezcla aleatoria y estratificacion.

Ademas, el etiquetado automatico deja **ruido**: ~2.225 textos aparecen con las dos
etiquetas a la vez. Se conservan (datos completos) y ponen un techo de performance.

## 3. Que muestra el EDA

![Longitud de tweets](imgs/02_longitud_tweets.png)

**La longitud no discrimina** (misma distribucion en ambas clases; el pico en 140 es
el limite historico de Twitter). La senial esta en el **vocabulario**:

![Top palabras por clase](imgs/02_top_palabras.png)

Y los **bigramas** ya muestran contexto con carga de sentimiento (`cant wait`,
`feel better`, `sorry hear`), lo que justifica vectorizar con unigramas + bigramas:

![Top bigramas por clase](imgs/02_top_bigramas.png)

## 4. Decisiones tomadas

| Decision | Motivo |
| --- | --- |
| Modelo **binario** (negativo/positivo) | El training no tiene ejemplos neutrales; no se puede aprender una clase sin datos |
| **TF-IDF (1-2 gramas, min_df=5)** | Eficiente para 1,6 M de textos cortos; los bigramas capturan negaciones (`not bad`) |
| **Logistic Regression** | Entrena en ~47 s, es interpretable (un peso por n-grama) y es un baseline fuerte para texto corto |
| Split 90/10 **estratificado con shuffle** | El archivo viene ordenado por clase |
| **No** remover stopwords; `can't` -> `cant` | Las negaciones (`not`, `no`) invierten el sentimiento; el tokenizador default las romperia |
| URLs -> `xxurl`, menciones -> `xxuser`, `#tag` -> `tag` | Conservar la senial estructural sin retener usuarios/dominios |
| Duplicados conflictivos: **se conservan** | La consigna exige datos completos; se documentan como limitacion |
| Modelo final **reentrenado con el 100%** (1.600.000) | Requisito mandatorio, verificado con `assert` |

## 5. Resultados

| Evaluacion | accuracy | precision | recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Validacion (160.000 tweets no vistos) | **0,8251** | 0,8195 | 0,8338 | 0,8266 |
| Test manual (359 tweets, otro dominio) | **0,8329** | 0,8050 | 0,8846 | 0,8429 |
| Training (el propio, chequeo de overfitting) | 0,8488 | | | |

- La brecha training/validacion es de solo ~2,4 puntos -> **no hay overfitting**
  (regularizacion L2 default + 1,44 M de ejemplos).
- El desempenio **se sostiene en el test etiquetado a mano sobre marcas y productos**:
  el modelo aprendio sentimiento genuino, no un artefacto del etiquetado por emoticones.

![Matriz de confusion validacion](imgs/05_confusion_validacion.png)

Errores casi simetricos entre clases, consistente con el balance del training.

## 6. Demo en vivo

El modelo final (guardado en `models/`) prediciendo tweets nuevos:

In [1]:
import sys
sys.path.insert(0, "notebooks")

import joblib
import pandas as pd
from utils import limpiar_tweets, VECTORIZER_JOBLIB, MODELO_JOBLIB

vec = joblib.load(VECTORIZER_JOBLIB)
lr = joblib.load(MODELO_JOBLIB)

ejemplos = [
    "I love this song, best concert ever!",
    "my phone died again and I lost all my photos",
    "cant wait to see you tomorrow!!",
    "not bad at all, actually pretty good",
    "I miss my dog so much... this house feels empty",
    "Great. Another monday. Yay...",          # sarcasmo: deberia fallar
]

proba = lr.predict_proba(vec.transform(limpiar_tweets(pd.Series(ejemplos))))[:, 1]
for texto, p in zip(ejemplos, proba):
    veredicto = "POSITIVO" if p >= 0.5 else "NEGATIVO"
    print(f"  p(pos)={p:.3f} -> {veredicto:8s} | {texto}")

  p(pos)=0.978 -> POSITIVO | I love this song, best concert ever!
  p(pos)=0.006 -> NEGATIVO | my phone died again and I lost all my photos
  p(pos)=0.972 -> POSITIVO | cant wait to see you tomorrow!!
  p(pos)=0.976 -> POSITIVO | not bad at all, actually pretty good
  p(pos)=0.006 -> NEGATIVO | I miss my dog so much... this house feels empty
  p(pos)=0.743 -> POSITIVO | Great. Another monday. Yay...


El ultimo ejemplo es **sarcasmo**: palabras literalmente positivas (`great`, `yay`)
en un mensaje negativo. El modelo cae en la trampa — y esa es exactamente la
limitacion estructural de una bolsa de palabras (seccion 8).

## 7. Interpretacion: que aprendio el modelo

![Coeficientes mas predictivos](imgs/06_coeficientes.png)

- Las palabras mas **predictivas** no son las mas **frecuentes**: `work` domina en
  frecuencia pero pesa poco; `sad`, `gutted`, `blessed` son inequivocas.
- **Tres de los terminos mas positivos contienen negaciones** (`cant wait`,
  `no problem`, `not bad`): la prueba de que conservar negaciones y usar bigramas
  fue la decision correcta.

### Similitud coseno (metrica vista en clase)

Mide el angulo entre los vectores TF-IDF de dos textos (1 = mismo vocabulario
ponderado, 0 = nada en comun), ignorando la longitud. Tres usos:

1. **Vecinos mas cercanos**: dado un tweet, recupera los que hablan de lo mismo.
2. **Ruido de etiquetas**: se hallaron tweets *identicos* (cos = 1.0) con etiquetas
   opuestas — evidencia directa del techo de performance.
3. **Centroides de clase**: cos(centroide negativo, centroide positivo) = **0,894**.
   Las dos clases hablan de lo mismo con casi el mismo vocabulario; el sentimiento
   vive en una fraccion chica de terminos. Por eso alcanza un modelo lineal.

### Topicos (NMF) por polaridad

- **Negativos**: trabajo/estudio, salud y dolor, extraniar a alguien, fallas tecnicas.
- **Positivos**: saludos y agradecimientos, humor, musica/eventos, festejos.

### Extension: tendencia por usuario

![Pesimismo observado por usuario](imgs/06_pesimismo_usuarios.png)

Sobre 6.245 usuarios con 20+ tweets: pico en 0 (usuarios que solo aparecen con
tweets positivos) y cola de usuarios siempre negativos. Se lee como **tendencia del
corpus** (el dataset se recolecto por emoticones), no como diagnostico de personas.

## 8. Limitaciones

1. **Sarcasmo e ironia**: ~1.100 errores de validacion son tweets negativos con
   vocabulario positivo (*"Great. History. Yay..."*). Invisible para bolsa de palabras.
2. **Ruido del etiquetado automatico**: tweets de agradecimiento etiquetados como
   negativos y textos identicos con las dos etiquetas. Ningun modelo puede acertar esos.
3. **Clase neutral ausente en training**: la aproximacion por umbral de incertidumbre
   (probabilidad cerca de 0,5 -> neutral) rinde poco (recall neutral 0,19):
   **"dudar" no es lo mismo que "ser neutral"** — hallazgo honesto de la extension.

![Confusion 3 clases por umbral](imgs/05_confusion_3clases.png)

## 9. Conclusiones

1. Un baseline clasico e interpretable (TF-IDF + Logistic Regression), entrenado con
   los **1.600.000 tweets completos**, clasifica polaridad con ~0,83 de accuracy y
   **generaliza a un dominio distinto** (test manual etiquetado a mano).
2. La senial de sentimiento es **lexica** y las **negaciones importan**: las
   decisiones de preprocesamiento se validaron en los propios coeficientes del modelo.
3. La **similitud coseno** mostro que las polaridades comparten casi todo el
   vocabulario y sirvio para probar la existencia de ruido de etiquetado.
4. **Mejora futura**: modelo de lenguaje preentrenado (embeddings contextuales) para
   capturar ironia y contexto, y datos neutrales reales para plantear 3 clases.

---

## Notebooks de detalle

| Notebook | Contenido |
| --- | --- |
| [01_carga_y_validacion](./notebooks/01_carga_y_validacion.ipynb) | Carga completa, clases, duplicados y ruido |
| [02_eda](./notebooks/02_eda.ipynb) | Longitud, vocabulario por clase, seniales de Twitter |
| [03_preprocesamiento](./notebooks/03_preprocesamiento.ipynb) | Limpieza documentada y persistencia |
| [04_entrenamiento](./notebooks/04_entrenamiento.ipynb) | TF-IDF + LR, split y modelo final con el 100% |
| [05_evaluacion](./notebooks/05_evaluacion.ipynb) | Metricas mandatorias, test manual, neutral por umbral |
| [06_interpretacion](./notebooks/06_interpretacion.ipynb) | Coeficientes, similitud coseno, topicos, sarcasmo, usuarios |